<a href="https://colab.research.google.com/github/Lianaulya/Practical-Statistics-for-Data-Scientists/blob/main/Chapter4_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## Identitas Mahasiswa

| | |
|---|---|
| **Nama** | Muammar Yaldi Fadhil |
| **NIM** | 101032300019 |
| **Kelas** | TK-47-03 |
| **Mata Kuliah** | Machine Learning |

---

# Chapter 4 – Regression and Prediction
**Book:** *Practical Statistics for Data Scientists* (O'Reilly)

Topics:
- Simple Linear Regression
- Multiple Linear Regression
- Evaluating Regression: R², RMSE, Residuals
- Cross-Validation
- Weighted Regression
- Prediction vs. Confidence Intervals
- Categorical (Dummy) Variables
- Multicollinearity & VIF
- Confounding Variables
- Polynomial and Spline Regression


In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm
import statsmodels.formula.api as smf

BASE_DATA = "https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data"
print("Libraries loaded!")

## 1. Simple Linear Regression

Simple linear regression fits a straight line $\hat{y} = b_0 + b_1 x$
that minimizes the **sum of squared residuals (RSS)**:

$$RSS = \sum_{i=1}^{n}(y_i - \hat{y}_i)^2 = \sum_{i=1}^{n}(y_i - b_0 - b_1 x_i)^2$$

The OLS (Ordinary Least Squares) solution:

$$b_1 = \frac{\sum(x_i - \bar{x})(y_i - \bar{y})}{\sum(x_i - \bar{x})^2} \qquad b_0 = \bar{y} - b_1\bar{x}$$

In [ ]:
# Lung disease data: PEFR vs. exposure to dust
lung = pd.read_csv(f"{BASE_DATA}/LungDisease.csv")
print(lung.head())
print(lung.describe())

In [ ]:
# Fit simple linear regression using statsmodels
model_simple = smf.ols("PEFR ~ Exposure", data=lung).fit()
print(model_simple.summary())

In [ ]:
# Plot regression line
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(lung["Exposure"], lung["PEFR"], alpha=0.6, color="steelblue", label="Data")
x_range = np.linspace(lung["Exposure"].min(), lung["Exposure"].max(), 100)
y_pred = model_simple.params[0] + model_simple.params[1] * x_range
ax.plot(x_range, y_pred, color="red", linewidth=2,
        label=f"OLS: PEFR = {model_simple.params[0]:.1f} + {model_simple.params[1]:.2f}·Exposure")
ax.set_title("Lung Disease — PEFR vs. Dust Exposure")
ax.set_xlabel("Years of Exposure")
ax.set_ylabel("PEFR (L/min)")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Multiple Linear Regression

With $p$ predictors:

$$\hat{y} = b_0 + b_1 x_1 + b_2 x_2 + \cdots + b_p x_p$$

In matrix form: $\hat{\mathbf{y}} = \mathbf{X}\hat{\boldsymbol{\beta}}$

OLS solution: $\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$

In [ ]:
# King County house sales
house = pd.read_csv(f"{BASE_DATA}/house_sales.csv", sep="\t")
print(f"Shape: {house.shape}")
print(house.head())

In [ ]:
# Multiple regression: price ~ sqft + bedrooms + bathrooms + floors + ...
features = ["SqFtTotLiving", "SqFtLot", "Bathrooms", "Bedrooms", "BldgGrade"]
X = house[features]
y = house["AdjSalePrice"]

model_multi = smf.ols(
    "AdjSalePrice ~ SqFtTotLiving + SqFtLot + Bathrooms + Bedrooms + BldgGrade",
    data=house
).fit()
print(model_multi.summary().tables[1])

## 3. Evaluating Regression: R², RMSE, and Residuals

**$R^2$ (Coefficient of Determination):** fraction of variance explained by the model.

$$R^2 = 1 - \frac{RSS}{TSS} = 1 - \frac{\sum(y_i-\hat{y}_i)^2}{\sum(y_i-\bar{y})^2}$$

**RMSE (Root Mean Squared Error):**

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

**Residual Analysis:** residuals $e_i = y_i - \hat{y}_i$ should be:
- Approximately normally distributed
- Homoscedastic (constant variance)
- Uncorrelated with predictors


In [ ]:
# R² and RMSE
y_pred = model_multi.predict(house[features])
residuals = y - y_pred
rmse = np.sqrt(mean_squared_error(y, y_pred))

print(f"R²:   {model_multi.rsquared:.4f}")
print(f"RMSE: ${rmse:,.0f}")

In [ ]:
# Residual plots
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Residuals vs. Fitted
axes[0].scatter(y_pred, residuals, alpha=0.3, color="steelblue", s=10)
axes[0].axhline(0, color="red", linewidth=1.5)
axes[0].set_title("Residuals vs. Fitted Values")
axes[0].set_xlabel("Fitted Values ($)")
axes[0].set_ylabel("Residuals ($)")

# Histogram of residuals
pd.Series(residuals).hist(bins=40, ax=axes[1], color="steelblue", edgecolor="black")
axes[1].set_title("Distribution of Residuals")
axes[1].set_xlabel("Residual ($)")

plt.tight_layout()
plt.show()

## 4. Cross-Validation

**K-Fold Cross-Validation** is used to estimate out-of-sample model performance:
1. Split data into $K$ equal folds
2. For each fold: train on $K-1$ folds, evaluate on the held-out fold
3. Average the $K$ performance metrics

This avoids overfitting to training data.

In [ ]:
# 5-fold cross-validation with sklearn
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

house_clean = house[features + ["AdjSalePrice"]].dropna()
X_cv = house_clean[features].values
y_cv = house_clean["AdjSalePrice"].values

lr = LinearRegression()
kf = KFold(n_splits=5, shuffle=True, random_state=42)

rmse_scores = []
r2_scores = []
for train_idx, test_idx in kf.split(X_cv):
    lr.fit(X_cv[train_idx], y_cv[train_idx])
    preds = lr.predict(X_cv[test_idx])
    rmse_scores.append(np.sqrt(mean_squared_error(y_cv[test_idx], preds)))
    ss_res = np.sum((y_cv[test_idx] - preds)**2)
    ss_tot = np.sum((y_cv[test_idx] - y_cv[test_idx].mean())**2)
    r2_scores.append(1 - ss_res/ss_tot)

print(f"5-Fold CV RMSE: ${np.mean(rmse_scores):,.0f} ± ${np.std(rmse_scores):,.0f}")
print(f"5-Fold CV R²:   {np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}")

## 5. Categorical Variables (Dummy Encoding)

Categorical predictors are encoded as **dummy (indicator) variables**.
For a categorical variable with $k$ levels, we create $k-1$ binary columns
to avoid perfect multicollinearity (the "dummy variable trap").

In [ ]:
# Add property type as dummy variable
house2 = house[features + ["PropertyType", "AdjSalePrice"]].dropna().copy()
print("Property Types:", house2["PropertyType"].unique())

# One-hot encoding
dummies = pd.get_dummies(house2["PropertyType"], drop_first=True)
print("Dummy columns:", list(dummies.columns))
house2 = pd.concat([house2, dummies], axis=1)
print(house2.head())

In [ ]:
# Regression with categorical variable
formula = ("AdjSalePrice ~ SqFtTotLiving + SqFtLot + Bathrooms + Bedrooms"
           " + BldgGrade + C(PropertyType)")
model_cat = smf.ols(formula, data=house2).fit()
print(model_cat.summary().tables[1])
print(f"\nWith PropertyType → R² = {model_cat.rsquared:.4f}")

## 6. Multicollinearity and VIF

**Multicollinearity** occurs when predictors are highly correlated.
It inflates standard errors and makes coefficients unstable.

**Variance Inflation Factor (VIF)**:

$$VIF_j = \frac{1}{1 - R^2_j}$$

where $R^2_j$ is obtained by regressing $x_j$ on all other predictors.
- $VIF \approx 1$: no multicollinearity
- $VIF > 5$ (or 10): problematic multicollinearity


In [ ]:
# Compute VIF for the house price model
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = house_clean[features].assign(const=1)
vif_data = pd.DataFrame()
vif_data["Feature"] = features
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(len(features))]
print(vif_data.sort_values("VIF", ascending=False).to_string(index=False))

## 7. Polynomial and Spline Regression

**Polynomial regression** adds polynomial terms to capture non-linear relationships:

$$\hat{y} = b_0 + b_1 x + b_2 x^2 + \cdots + b_d x^d$$

**Spline regression** fits piecewise polynomials joined at **knot points**,
allowing more flexible curves without extreme values at the edges.

In [ ]:
# Polynomial regression: house price vs. living area
from numpy.polynomial.polynomial import polyfit

house_small = house_clean.sample(3000, random_state=42)
x = house_small["SqFtTotLiving"].values
y_h = house_small["AdjSalePrice"].values

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, degree, color in zip(axes, [1, 3], ["red", "green"]):
    ax.scatter(x, y_h, alpha=0.15, s=10, color="steelblue")
    coeffs = np.polyfit(x, y_h, degree)
    x_line = np.linspace(x.min(), x.max(), 200)
    y_line = np.polyval(coeffs, x_line)
    ax.plot(x_line, y_line, color=color, linewidth=2,
            label=f"Degree-{degree} polynomial")
    ax.set_title(f"Polynomial Regression (degree={degree})")
    ax.set_xlabel("SqFt Living Area")
    ax.set_ylabel("Sale Price ($)")
    ax.legend()

plt.tight_layout()
plt.show()

## Summary

| Concept | Key Formula |
|---------|------------|
| Simple OLS | $\hat{y} = b_0 + b_1 x$ |
| Multiple OLS | $\hat{\mathbf{y}} = \mathbf{X}\hat{\boldsymbol{\beta}}$ |
| R² | $1 - RSS/TSS$ |
| RMSE | $\sqrt{MSE}$ |
| CV | K-fold out-of-sample evaluation |
| VIF | $1/(1-R^2_j)$ — multicollinearity |\n